In [1]:
import torch
from PIL import Image
from torchvision import transforms
import torch.nn.functional as F

# 1. DINOv2 로드 (small이 빠름, base가 더 정확)
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
model.eval().cuda()

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def get_patch_features(img):
    x = preprocess(img).unsqueeze(0).cuda()
    with torch.no_grad():
        out = model.forward_features(x)
    # DINOv2: 'x_norm_patchtokens' 키에 patch token이 있음
    return out['x_norm_patchtokens'].squeeze(0)  # (N, D)

# 2. 증강 함수
def brighten(img, factor):
    return transforms.functional.adjust_brightness(img, factor)

# 3. shift 분석
def analyze_shift(f_orig, f_aug):
    delta = f_aug - f_orig  # (N, D)
    
    # mean direction
    v_mean = delta.mean(dim=0)
    v_mean_norm = v_mean / (v_mean.norm() + 1e-8)
    
    # per-patch alignment with mean direction
    delta_norm = delta / (delta.norm(dim=1, keepdim=True) + 1e-8)
    cos = (delta_norm @ v_mean_norm)  # (N,)
    
    # variance ratio via SVD
    U, S, _ = torch.linalg.svd(delta - delta.mean(0), full_matrices=False)
    var_ratio = (S[0]**2) / (S**2).sum()
    
    # magnitude map
    mags = delta.norm(dim=1)  # (N,)
    
    return {
        'alignment_mean': cos.mean().item(),
        'alignment_std': cos.std().item(),
        'first_pc_ratio': var_ratio.item(),
        'magnitude_mean': mags.mean().item(),
        'magnitude_std': mags.std().item(),
        'magnitude_map': mags.cpu().numpy(),  # 16x16 reshape해서 시각화
    }

# 4. 실험 루프
img = Image.open('test.jpg').convert('RGB')
f_orig = get_patch_features(img)

for name, aug_fn in [
    ('bright_0.6', lambda x: brighten(x, 0.6)),
    ('bright_1.4', lambda x: brighten(x, 1.4)),
    ('contrast_0.7', lambda x: transforms.functional.adjust_contrast(x, 0.7)),
    # ... local paste, rotation 등 추가
]:
    img_aug = aug_fn(img)
    f_aug = get_patch_features(img_aug)
    result = analyze_shift(f_orig, f_aug)
    print(name, result)

Using cache found in /Users/song-inseop/.cache/torch/hub/facebookresearch_dinov2_main
/Users/song-inseop/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/Users/song-inseop/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/Users/song-inseop/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


AssertionError: Torch not compiled with CUDA enabled